In [3]:
import asyncio
import json

from dotenv import load_dotenv
from openai import AsyncOpenAI
from openai.types.chat import ChatCompletionMessage, ChatCompletionMessageToolCall

from navgent.browser.browser_manager import BrowserManager
from navgent.browser.browser_session import BrowserSession
from navgent.context import ToolContext
from navgent.tool_schemas import TOOL_SCHEMAS
from navgent.tools import TOOLS

MODEL = "gpt-4.1-mini"
MAX_STEPS = 3

SYSTEM_PROMPT = """
You are a browser agent. Use the tools to complete the task.
When the task is finished, reply with your answer instead of calling a tool.
"""


def build_prompt(events: list[str]) -> list[dict]:
    history = "\n\n".join(events)
    user_message = f"{history}\n\nWhat is the next step?"
    print(f"user_message: {user_message}")
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]


async def ask_model(client: AsyncOpenAI, events: list[str]) -> ChatCompletionMessage:
    response = await client.chat.completions.create(
        model=MODEL, messages=build_prompt(events), tools=TOOL_SCHEMAS, parallel_tool_calls=False
    )
    return response.choices[0].message


async def execute_tool_call(context: ToolContext, tool_call: ChatCompletionMessageToolCall) -> str:
    print(tool_call.function.name, tool_call.function.arguments)
    tool = TOOLS[tool_call.function.name]
    arguments = json.loads(tool_call.function.arguments)
    return await tool(context, **arguments)


def result_to_event(tool_call: ChatCompletionMessageToolCall, tool_output: str) -> str:
    name = tool_call.function.name
    return f"<{name} arguments={tool_call.function.arguments}>\n{tool_output}\n</{name}>"


async def run(task: str) -> str | None:
    load_dotenv()
    client = AsyncOpenAI()
    browser_manager = BrowserManager(headless=False)
    browser_session = BrowserSession(browser_manager)
    context = ToolContext(browser_session=browser_session)
    events = [f"<task>\n{task}\n</task>"]

    try:
        for _ in range(MAX_STEPS):
            message = await ask_model(client, events)
            if not message.tool_calls:
                return message.content

            tool_call = message.tool_calls[0]
            tool_output = await execute_tool_call(context, tool_call)
            events.append(result_to_event(tool_call, tool_output))
        return None
    finally:
        await browser_session.close()
        await browser_manager.stop()


task = "Go to https://www.saucedemo.com/ and login."

In [4]:
answer = await run(task)
print("Answer:", answer)

user_message: <task>
Go to https://www.saucedemo.com/ and login.
</task>

What is the next step?
navigate {"url":"https://www.saucedemo.com/"}
user_message: <task>
Go to https://www.saucedemo.com/ and login.
</task>

<navigate arguments={"url":"https://www.saucedemo.com/"}>
<page url="https://www.saucedemo.com/" title="Swag Labs" pages_below="0.0">

Interactive elements:
[0] <input type=text empty> Username
[1] <input type=password empty> Password
[2] <input type=submit> Login

</page>
<page-content>iframe[0] Swag Labs Accepted usernames are: standard_user locked_out_user problem_user performance_glitch_user error_user visual_user Password for all users: secret_sauce</page-content>
<available-tabs>[Tab(id='tab:0', selected=True, title='Swag Labs', url='https://www.saucedemo.com/')]</available-tabs>
<available-iframes>
frame:0 [selected] [main] url='https://www.saucedemo.com/'
</available-iframes>
</navigate>

What is the next step?
execute_steps {"steps":[{"action":"fill","element_inde